In [9]:
import os
import json
import time
import random

import cv2
import glob
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import numpy as np
import torch
from vidgear.gears import WriteGear

from koger_detection.obj_det.mydatasets import VideoDataset
from koger_detection.obj_det.predictors import Predictor

In [10]:
load_dotenv()

True

In [11]:

video_folder = os.environ.get("VIDEO_FOLDER")

out_folder = os.path.join(video_folder, "processing", "annotation-search")
os.makedirs(out_folder, exist_ok=True)


create_video = False

In [12]:
run_folder = os.path.join(os.environ.get("PROJECT_ROOT"), "models", "runs", "main")
run_name = "02-24-2026-16-34-06"
cfg_file = os.path.join(run_folder, run_name, "cfg.json") 

with open(cfg_file, "r") as f:
    cfg = json.load(f)
    
cfg_m = cfg['model']
cfg_m['model_weights_pth'] = os.path.join(run_folder, run_name, "final_model.pth")
cfg_m['box_detections_per_img'] = 512
cfg_m['rpn_pre_nms_top_n_test'] = 2000
cfg_m['rpn_post_nms_top_n_test'] = 1000

In [13]:

predictor = Predictor(cfg_m)

In [14]:
camera_names = ["cam01-bear_outflow", "cam02-confluence-big", "cam05-airport_outflow", "cam07-grass_outflow", "cam09-trail_outflow"] 

all_video_files = []

for camera_name in camera_names:
    video_files = sorted(glob.glob(os.path.join(video_folder, camera_name, "Mp4Record", "*",  "RecM0A*.mp4")))
    all_video_files.extend(random.sample(video_files, 40))

    peak_density_videos = []
    for video_file in video_files:
        video_name = os.path.basename(video_file)
        vid_time = int(video_name.split("_")[2])
        if (vid_time > 163219) and (vid_time < 203259):
            peak_density_videos.append(video_file)
    all_video_files.extend(random.sample(peak_density_videos, 40))
         

    
print(f"found {len(all_video_files)} video files")

found 400 video files


In [15]:
for video_ind, video_file in enumerate(all_video_files):
    video_name = os.path.basename(video_file).split(".")[0]

    ds = VideoDataset(video_file)
    data_loader = torch.utils.data.DataLoader(ds, batch_size=1, num_workers=0)

    if create_video:
        output_params = {"-input_framerate": 30,
                        "-pix_fmt": "yuv420p"}
        writer = WriteGear(output=os.path.join(out_folder, out_name),
                        **output_params) 


    all_boxes = []
    all_scores = []
    all_labels = []

    t0 = time.time()
    first_frame = 0
    max_im = None # 10000
    for ind, f in enumerate(data_loader):
        if ind < first_frame:
            continue
        if max_im is not None:
            if ind >= max_im:
                break
        if ind % 1000 == 0:
            print(f"Frame number {ind} processed.")
        res = predictor(f[0])
        boxes = res['boxes'].to('cpu').numpy().astype(np.uint32)
        scores = res['scores'].to('cpu').numpy()
        labels = res['labels'].to('cpu').numpy()

        image_name = f"{video_name}-frame-{ind}"

        all_boxes.append(boxes)
        all_scores.append(scores)
        all_labels.append(labels)

        if create_video:
            frame = f[0].numpy().copy() # Copy makes circle work for unclear reasons
            
            for box, score, label in zip(boxes, scores, labels):
                if score < .9:
                    continue
                x = np.mean([box[0], box[2]])
                y = np.mean([box[1], box[3]])
                if label == 1:
                    cv2.circle(frame, [int(x), int(y)], 4, (4,217,255), -1)
                else:
                    cv2.circle(frame, [int(x), int(y)], 8, (255, 0, 0), -1)
            if frame is not None:
                writer.write(frame, rgb_mode=True)
            else:
                print("skipping.")
    ds.stop()   
    if create_video: 
        writer.close()
    total_time = time.time() - t0
    print(video_ind, total_time, f"fps: {ind / total_time}", f"{ind} frames processed.")

    np.savez_compressed(os.path.join(out_folder, f"{video_name}-boxes.npz"), *all_boxes)
    np.savez_compressed(os.path.join(out_folder, f"{video_name}-scores.npz"), *all_scores)
    np.savez_compressed(os.path.join(out_folder, f"{video_name}-labels.npz"), *all_labels)

Frame number 0 processed.
Frame number 1000 processed.
Empty frame. Ending stream.
FileVideoStream stopped.
152.41622591018677 fps: 11.83601017035437 1804 frames processed.
Frame number 0 processed.
Frame number 1000 processed.
Frame number 2000 processed.
Empty frame. Ending stream.
FileVideoStream stopped.
202.4343774318695 fps: 12.10268745398524 2450 frames processed.
Frame number 0 processed.
Frame number 1000 processed.
Frame number 2000 processed.
Empty frame. Ending stream.
FileVideoStream stopped.
205.3100609779358 fps: 12.123127274641876 2489 frames processed.
Frame number 0 processed.
Frame number 1000 processed.
Frame number 2000 processed.
Empty frame. Ending stream.
FileVideoStream stopped.
210.40775656700134 fps: 11.85317519036339 2494 frames processed.
Frame number 0 processed.
Frame number 1000 processed.
Frame number 2000 processed.
Empty frame. Ending stream.
FileVideoStream stopped.
203.92482733726501 fps: 12.205478030801611 2489 frames processed.
Frame number 0 proc

In [ ]:
from yolox.tracker.byte_tracker import BYTETracker

In [ ]:
class BearArguments:
    def __init__(self):
        self.track_thresh = 0.6
        self.track_buffer = 30
        self.match_thresh = 0.8
        self.min_box_area = 10
        self.frame_rate = 30
        self.mot20 = False
        self.track_id = True
        self.nms = 0.65

bear_args = BearArguments()

class SalmonArguments:
    def __init__(self):
        self.track_thresh = 0.85
        self.track_buffer = 3
        self.match_thresh = 0.8
        self.min_box_area = 10
        self.frame_rate = 30
        self.mot20 = False
        self.track_id = True
        self.nms = 0.65

salmon_args = SalmonArguments()

In [ ]:
args.track_thresh

In [ ]:
import pickle

In [ ]:
tracker = BYTETracker(bear_args)

boxes = []
z_boxes = np.load(os.path.join(out_folder, f"{video_name}-boxes.npz"))
z_scores = np.load(os.path.join(out_folder, f"{video_name}-scores.npz"))
z_labels = np.load(os.path.join(out_folder, f"{video_name}-labels.npz"))

tracks = {}

class_to_track = 1 # 1 is bear, 2 is salmon

frame_num = 0
for boxes_key, scores_key, labels_key in zip(z_boxes, z_scores, z_labels):
    dets = np.concatenate([z_boxes[boxes_key], 
                           z_scores[scores_key][:, np.newaxis], 
                           z_scores[scores_key][:, np.newaxis],
                           z_labels[labels_key][:, np.newaxis]
                           ], 
                           axis=1)
    dets = dets[dets[:,-1]==class_to_track]
    online_targets = tracker.update(torch.from_numpy(dets), [2160, 3840], [2160, 3840])

    for strack in online_targets:
        if strack.is_activated:
            track_id = strack.track_id
            if track_id not in tracks:
                tracks[track_id] = {}
            tracks[track_id][strack.frame_id] = strack.tlwh

with open(os.path.join(out_folder, f'{video_name}-tracks-class-{class_to_track}.pkl'), 'wb') as outp:
    pickle.dump(tracks, outp, pickle.HIGHEST_PROTOCOL)


In [ ]:
with open(os.path.join(out_folder, f'{video_name}-tracks.pkl'), 'wb') as outp:
    pickle.dump(tracks, outp, pickle.HIGHEST_PROTOCOL)

In [ ]:
test = pickle.load(open(os.path.join(out_folder, f'{video_name}-tracks.pkl'), 'rb'))


In [ ]:
tracks_length = []
for track_id, track in tracks.items():
    if len(track) > 10:
        tracks_length.append(len(track))
plt.hist(tracks_length, bins=100)


In [ ]:

import matplotlib as mpl

In [ ]:
cmap = mpl.colormaps['tab20']
colors = [(np.array(c) * 255) for c in cmap.colors]


In [ ]:
ds = VideoDataset(video_file)
data_loader = torch.utils.data.DataLoader(ds, batch_size=1, num_workers=0)
out_name = "test-track-2.mp4"

output_params = {"-input_framerate": 30,
                "-pix_fmt": "yuv420p"}
writer = WriteGear(output=os.path.join(out_folder, out_name),
                **output_params) 

t0 = time.time()
first_frame = 0
max_im = None # 10000
for ind, f in enumerate(data_loader):
    if ind < first_frame:
        continue
    if max_im is not None:
        if ind >= max_im:
            break
    if ind % 1000 == 0:
        print(f"Frame number {ind} processed.")

    frame = f[0].numpy().copy() # Copy makes circle work for unclear reasons

    for track_id, track in tracks.items():
        if ind in track:
            box = track[ind]
            x = box[0] + int(box[2] / 2)
            y = box[1] + int(box[3] / 2)
            color = colors[track_id % len(colors)]
            cv2.circle(frame, [int(x), int(y)], 8, (int(color[0]), int(color[1]), int(color[2])), -1)
            cv2.putText(frame, str(track_id), (int(x), int(y)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
    
    if frame is not None:
        writer.write(frame, rgb_mode=True)
    else:
        print("skipping.")

    if ind > 20000:
        break
ds.stop()   

writer.close()
total_time = time.time() - t0
print(total_time, f"fps: {ind / total_time}", f"{ind} frames processed.")

In [ ]:
type(color[0]), color[1], color[2]

In [ ]:
tracks.keys()

In [ ]:
create_video = True
for video_file in video_names:
    video_name = os.path.splitext(os.path.basename(video_file))[0]
    out_name = video_name # probably needs .mp4
    ds = VideoDataset(video_file)
    data_loader = torch.utils.data.DataLoader(ds, batch_size=1, num_workers=0)

    if create_video:
        output_params = {"-input_framerate": 30,
                    "-pix_fmt": "yuv420p"}
        writer = WriteGear(output=os.path.join(out_folder, out_name),
                        **output_params) 


    all_boxes = []
    all_scores = []

    t0 = time.time()
    first_frame = 0
    max_im = None # 10000
    for ind, f in enumerate(data_loader):
        if ind < first_frame:
            continue
        if max_im is not None:
            if ind >= max_im:
                break
        if ind % 1000 == 0:
            print(f"Frame number {ind} processed.")
        res = predictor(f[0])
        boxes = res['boxes'].to('cpu').numpy().astype(np.uint32)
        scores = res['scores'].to('cpu').numpy()
        labels = res['labels'].to('cpu').numpy()

        image_name = f"{video_name}-frame-{ind}"

        all_boxes.append(boxes)
        all_scores.append(scores)

        if create_video:
            frame = f[0].numpy().copy() # Copy makes circle work for unclear reasons
            
            for box, score, label in zip(boxes, scores, labels):
                if score < .9:
                    continue
                x = np.mean([box[0], box[2]])
                y = np.mean([box[1], box[3]])
                if label == 1:
                    cv2.circle(frame, [int(x), int(y)], 4, (4,217,255), -1)
                else:
                    cv2.circle(frame, [int(x), int(y)], 8, (255, 0, 0), -1)
            if frame is not None:
                writer.write(frame[:720, :1280], rgb_mode=True)
            else:
                print("skipping.")
    ds.stop()   
    if create_video: 
        writer.close()
    total_time = time.time() - t0
    print(total_time, f"fps: {ind / total_time}", f"{ind} frames processed.")

    np.savez_compressed(os.path.join(out_folder, f"{video_name}-boxes.npz"), *all_boxes)
    np.savez_compressed(os.path.join(out_folder, f"{video_name}-scores.npz"), *all_scores)

In [ ]:
output_params = {"-input_framerate": 30,
# }
                "-pix_fmt": "yuv420p"}
writer = WriteGear(output=os.path.join(out_folder, out_name),
                **output_params) 

for i in range(300):
    writer.write(np.ones((400, 400, 3)), rgb_mode=True)

writer.close()



In [ ]:
np.savez_compressed(os.path.join(out_folder, f"{video_name}-allboxes.npz"), *all_boxes)
np.savez_compressed(os.path.join(out_folder, f"{video_name}-allscores.npz"), *all_scores)

In [ ]:
os.path.join(out_folder, f"{video_name}-allboxes.npz")

In [ ]:
boxes = []
z = np.load(os.path.join(out_folder, f"{video_name}-boxes.npz"))
for key in z:
    boxes.append(z[key])

scores = []
z = np.load(os.path.join(out_folder, f"{video_name}-scores.npz"))
for key in z:
    scores.append(z[key])


In [ ]:
thresh = .7
counts = []
for frame_boxes, frame_scores in zip(boxes, scores):
    mask = frame_scores > thresh
    count = np.sum(mask)
    counts.append(count)

In [ ]:
plt.plot(counts)

In [ ]:
scores[0] >.8